# 03 · Training, and why the *change* to `W` is small

**Recap so far:**
- 01: a layer is `output = W @ x`; `W` is the model's knobs; **fine-tuning nudges them**.
- 02: a matrix can be **low rank** (few real directions), and a low-rank matrix can be
  written compactly as **`B · A`**.

This notebook connects them: we'll see what "training" actually does, and then the key
insight — **the change we make to `W` is usually low rank.** Let's build it up.

In [ ]:
import numpy as np
np.random.seed(1)

## Step 1 — Training = nudge a knob to reduce the error

Forget matrices for a second. Take **one** knob `w`. We feed in `x = 2` and we *want*
the output `w · x` to equal `y = 10`. So the perfect knob is `w = 5` (since 5×2 = 10).

We start with a bad guess `w = 1` and improve it. The recipe ("gradient descent") is:

1. see how wrong we are: `error = w·x − y`
2. nudge `w` a **small step** in the direction that shrinks the error
3. repeat

Watch `w` crawl toward 5:

In [ ]:
x, y = 2.0, 10.0     # we want w*x == y, so the perfect w is 5
w = 1.0              # a deliberately bad starting guess
lr = 0.05            # how big each nudge is

for step in range(20):
    pred  = w * x
    error = pred - y
    grad  = 2 * error * x      # the direction that increases error^2...
    w    -= lr * grad          # ...so step the OPPOSITE way (downhill)

print("learned w =", round(w, 4), " (perfect value is 5)")

That's all training is: **measure the error, take small steps to reduce it.** A real
model does this for millions of knobs at once, but the idea is exactly this.

## Step 2 — The "change" to a weight matrix has a name: ΔW

When we fine-tune, we start from a pretrained matrix `W` and end at a new one. The
**difference** between them is itself a matrix — the matrix of changes. We call it
**ΔW** ("delta W", literally *change in W*):

$$W_{\text{new}} = W_{\text{old}} + \Delta W$$

In [ ]:
W_old = np.array([[3, 1],
                  [0, 2]])
dW    = np.array([[0, 0],
                  [1, 0]])     # we nudged ONE knob

W_new = W_old + dW
print("W_new = W_old + ΔW =\n", W_new)

## Step 3 — Frozen vs trainable

We don't have to change every knob. We can **freeze** `W_old` (leave it exactly as is)
and only **learn the change `ΔW` separately**. "Frozen" just means: that number gets no
nudge in Step 1's loop. "Trainable" means it does.

This is the setup LoRA will use: **keep the giant pretrained `W` frozen, and learn only
the change.** Which raises the money question...

## Step 4 — How big does ΔW need to be? (Here's the magic)

Does `ΔW` have to be a big, complicated, full-rank matrix? Let's test it.

We take a pretrained `W` (50×50 = 2,500 knobs) and define a small **task**: make the
layer turn a few specific inputs into a few specific desired outputs. Then we compute the
**smallest change `ΔW`** that does the job, and measure its **rank** (from notebook 02).

In [ ]:
d = k = 50
W = np.random.randn(d, k)          # pretrained layer: 2,500 knobs

print(f"{'# of goals in the task':>24} {'rank of the change ΔW':>24}")
for n_goals in (2, 5, 10):
    X = np.random.randn(k, n_goals)         # the task's inputs
    Y = np.random.randn(d, n_goals)         # the desired outputs
    dW = (Y - W @ X) @ np.linalg.pinv(X)    # smallest ΔW with (W+ΔW)X = Y
    print(f"{n_goals:>24} {np.linalg.matrix_rank(dW):>24}")

Look at that: **the rank of `ΔW` exactly equals the number of goals.** A task with only a
few things to learn needs only a **low-rank** change — even though `W` itself is huge.

(Honest note: this is exact for a single linear layer. For a deep network it's the
*intuition*, and notebook 04 ends by pointing at the real experiment that tests it.)

## Recap

- **Training** = measure the error, nudge knobs in small steps to reduce it.
- The change to a weight is a matrix **`ΔW`**, and `W_new = W_old + ΔW`.
- We can **freeze** `W_old` and learn only `ΔW`.
- For a task with few new things to learn, **`ΔW` is low rank.**

And from notebook 02 we know a low-rank matrix = **`B · A`**. So instead of learning the
whole `ΔW`, we can just learn a small `B` and `A`. **That is LoRA** — assembled next.

**BAM!** Next: **`04 — LoRA, assembled`.**